# Dataset Generator – Synthetic Customer Data

This notebook generates a synthetic dataset of customer records, suitable for:
- Classification tasks (binary target: `will_buy_again`)
- Exploratory data analysis (EDA) practice
- Dashboard demos

**Columns include:**
- Demographics: Age, Gender, Marital Status, Country
- Financial: Annual Income, Credit Rating
- Behavioural: Purchase Frequency, Satisfaction Score, Days since last purchase
- Identifiers: Customer ID, Name, Email, Postal Code
- Target: `will_buy_again` (1 = likely to repurchase)

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)
fake = Faker()
Faker.seed(42)

# Configuration
N_RECORDS = 1000  # Number of customers to generate
MISSING_PROBABILITY = 0.05  # Probability of a value being missing (where applicable)

In [2]:
def generate_customer_data(n: int, missing_prob: float = 0.05) -> pd.DataFrame:
    """
    Generate a synthetic customer dataset.
    
    Args:
        n: Number of records.
        missing_prob: Probability of inserting a missing value (pd.NA) for certain columns.
    
    Returns:
        pandas.DataFrame with customer data and a binary target.
    """
    
    # Helper to randomly insert missing values
    def maybe_missing(value):
        return pd.NA if np.random.random() < missing_prob else value

    # Generate base data
    data = {
        'customer_id': [f'CUST_{i:05d}' for i in range(1, n+1)],
        'full_name': [fake.name() for _ in range(n)],
        'age': [maybe_missing(np.random.randint(18, 81)) for _ in range(n)],
        'gender': [maybe_missing(np.random.choice(['M', 'F', 'Other'])) for _ in range(n)],
        'marital_status': [maybe_missing(np.random.choice(
            ['Single', 'Married', 'Divorced', 'Widowed'], 
            p=[0.45, 0.35, 0.15, 0.05]
        )) for _ in range(n)],
        'annual_income': [maybe_missing(round(np.random.normal(60000, 30000), 0)) for _ in range(n)],
        'num_children': [maybe_missing(np.random.choice([0, 1, 2, 3, 4], p=[0.3, 0.3, 0.2, 0.15, 0.05])) for _ in range(n)],
        'country': [np.random.choice(['Mexico', 'Colombia', 'Argentina', 'Chile']) for _ in range(n)],
        'registration_date': [fake.date_between(start_date='-5y', end_date='today') for _ in range(n)],
        'purchase_frequency': [maybe_missing(round(np.random.uniform(0.5, 20.0), 1)) for _ in range(n)],
        'satisfaction_score': [maybe_missing(round(np.random.uniform(0.0, 10.0), 1)) for _ in range(n)],
        'preferred_product': [np.random.choice(['Electronics', 'Clothing', 'Home', 'Sports', 'Beauty']) for _ in range(n)],
        'email': [fake.email() for _ in range(n)],
        'avg_discount': [maybe_missing(np.random.randint(0, 51)) for _ in range(n)],
        'is_active': [maybe_missing(np.random.choice([True, False], p=[0.7, 0.3])) for _ in range(n)],
        'postal_code': [fake.postcode() for _ in range(n)],
        'customer_comments': [fake.sentence() for _ in range(n)],
        'days_since_last_purchase': [maybe_missing(np.random.randint(0, 1825)) for _ in range(n)],
        'credit_rating': [np.random.choice(['Low', 'Medium', 'High', 'Premium'], p=[0.25, 0.35, 0.25, 0.15]) for _ in range(n)],
    }
    
    df = pd.DataFrame(data)
    
    # --- Create target column (will_buy_again) ---
    # Simulate logic based on other features to make it realistic
    # Higher satisfaction, higher income, more frequent purchases -> more likely to buy again
    prob_buy = (
        0.3 + 
        0.3 * (df['satisfaction_score'].fillna(5) / 10) + 
        0.2 * (df['annual_income'].fillna(50000) / 150000) +
        0.2 * (1 - df['days_since_last_purchase'].fillna(900) / 1800)
    )
    # Clip probabilities between 0 and 1
    prob_buy = np.clip(prob_buy, 0.0, 1.0)
    
    df['will_buy_again'] = np.random.binomial(1, prob_buy)
    
    return df

In [3]:
# Generate the dataset
df_customers = generate_customer_data(N_RECORDS, MISSING_PROBABILITY)

# Show first few rows
df_customers.head()

C:\Users\phabe\AppData\Local\Temp\ipykernel_43688\824608822.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  0.3 * (df['satisfaction_score'].fillna(5) / 10) +
C:\Users\phabe\AppData\Local\Temp\ipykernel_43688\824608822.py:51: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  0.2 * (df['annual_income'].fillna(50000) / 150000) +
C:\Users\phabe\AppData\Local\Temp\ipykernel_43688\824608822.py:52: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) i

,customer_id,full_name,age,gender,marital_status,annual_income,num_children,country,registration_date,purchase_frequency,satisfaction_score,preferred_product,email,avg_discount,is_active,postal_code,customer_comments,days_since_last_purchase,credit_rating,will_buy_again
0,CUST_00001,Allison Hill,56,Other,Married,<NA>,2,Colombia,2025-09-11,<NA>,7.8,Clothing,lisa57@example.com,29,True,52829,Mission worry goal Mrs decide between.,1036,Medium,1
1,CUST_00002,Noah Rhodes,32,Other,Single,<NA>,0,Mexico,2023-10-16,<NA>,<NA>,Electronics,shawdon@example.org,33,False,79779,Down could feel.,204,High,1
2,CUST_00003,Angie Henderson,78,F,Single,99335.0,1,Mexico,2022-04-24,6.9,9.3,Home,chrisschneider@example.org,28,True,32854,Whatever own democratic grow campaign.,39,Low,1
3,CUST_00004,Daniel Wagner,75,<NA>,Married,53270.0,0,Chile,2026-03-01,2.4,2.0,Clothing,jacobcox@example.net,38,False,69426,Instead near end power clear lay war put.,1552,High,1
4,CUST_00005,Cristian Santos,28,Other,Divorced,85788.0,1,Mexico,2024-10-31,9.3,9.1,Home,guerrerorobert@example.com,4,True,43609,Movie audience run yet.,824,High,1


In [ ]:
# Uncomment to save the dataset
# df_customers.to_csv('customer_data.csv', index=False)
# print("Dataset saved as 'customer_data.csv'")

## Summary

- Generated **{N_RECORDS}** synthetic customer records.
- The dataset includes **20 columns** with a mix of categorical, numerical, and text data.
- A realistic **binary target** (`will_buy_again`) was created using a probabilistic function based on satisfaction, income, and recency.
- Missing values (~5%) were intentionally introduced to simulate real‑world data imperfections.

**Next Steps:**
- Perform EDA on this dataset.
- Train a classification model to predict `will_buy_again`.
- Use this as a benchmark for feature engineering exercises.

In [4]:
print("=== Dataset Info ===")
df_customers.info()

print("\n=== Missing Values ===")
print(df_customers.isnull().sum())

print("\n=== Target Distribution ===")
print(df_customers['will_buy_again'].value_counts(normalize=True))

=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               1000 non-null   object
 1   full_name                 1000 non-null   object
 2   age                       935 non-null    object
 3   gender                    952 non-null    object
 4   marital_status            950 non-null    object
 5   annual_income             952 non-null    object
 6   num_children              946 non-null    object
 7   country                   1000 non-null   object
 8   registration_date         1000 non-null   object
 9   purchase_frequency        949 non-null    object
 10  satisfaction_score        952 non-null    object
 11  preferred_product         1000 non-null   object
 12  email                     1000 non-null   object
 13  avg_discount              949 non-null    object
 14  is_a